<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/30_production_rag_system/production_rag_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers pandas numpy scikit-learn

In [2]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

In [3]:
documents = [
    "Elon Musk founded SpaceX.",
    "SpaceX works on rockets and space exploration.",
    "Tesla builds electric cars.",
    "Elon Musk is CEO of Tesla."
]

df = pd.DataFrame({"text": documents})

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(df["text"].tolist())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
chat_history = []
feedback_memory = []

In [6]:
def retrieve(query, k=2):
    query_embedding = model.encode(query)
    scores = util.cos_sim(query_embedding, doc_embeddings)[0]
    scores = scores.cpu().numpy() if hasattr(scores, "cpu") else np.array(scores)

    top_idx = np.argsort(-scores)[:k]
    return df.iloc[top_idx], scores[top_idx]

In [7]:
def needs_calculation(query):
    return any(op in query for op in ["+", "-", "*", "/"])

def calculator(query):
    try:
        return str(eval(query))
    except:
        return None

In [8]:
answers = ["Elon Musk", "SpaceX", "Tesla"]
answer_embeddings = model.encode(answers)

def extract_answer(query):
    query_embedding = model.encode(query)
    scores = util.cos_sim(query_embedding, answer_embeddings)[0]
    scores = scores.cpu().numpy() if hasattr(scores, "cpu") else np.array(scores)

    return answers[np.argmax(scores)]

In [9]:
def get_confidence(scores):
    score = float(scores[0])

    if score > 0.7:
        return "High", score
    elif score > 0.5:
        return "Medium", score
    else:
        return "Low", score

In [10]:
def check_feedback(query):
    for q, ans in feedback_memory:
        if q.lower() == query.lower():
            return ans
    return None

def store_feedback(query, predicted, true):
    if true and predicted.lower() != true.lower():
        feedback_memory.append((query, true))

In [11]:
def production_rag(query, true_answer=None):
    print("\n🔹 Query:", query)

    # Feedback check
    learned = check_feedback(query)
    if learned:
        print("♻️ Using learned answer")
        return learned

    # Tool
    calc_result = None
    if needs_calculation(query):
        expr = "".join([c for c in query if c in "0123456789+-*/"])
        calc_result = calculator(expr)
        print("🧮 Tool Output:", calc_result)

    # Retrieval
    docs, scores = retrieve(query)
    print("\n📄 Retrieved:\n", docs)

    # Answer
    answer = extract_answer(query)

    # Confidence
    conf, score = get_confidence(scores)

    print("\n✅ Answer:", answer)
    print("📊 Confidence:", conf, score)

    # Store feedback
    store_feedback(query, answer, true_answer)

    chat_history.append((query, answer))

    return answer

In [12]:
production_rag("Who started SpaceX?", "Elon Musk")
production_rag("Who started SpaceX?")
production_rag("What is 25*4 and which company works on rockets?")


🔹 Query: Who started SpaceX?

📄 Retrieved:
                                              text
0                       Elon Musk founded SpaceX.
1  SpaceX works on rockets and space exploration.

✅ Answer: SpaceX
📊 Confidence: High 0.761458158493042

🔹 Query: Who started SpaceX?
♻️ Using learned answer

🔹 Query: What is 25*4 and which company works on rockets?
🧮 Tool Output: 100

📄 Retrieved:
                                              text
1  SpaceX works on rockets and space exploration.
0                       Elon Musk founded SpaceX.

✅ Answer: SpaceX
📊 Confidence: Low 0.4980468153953552


'SpaceX'